## Install `holidays`

The Databricks Serverless environment does not ship the `holidays` package. This cell installs it and restarts Python so `src.features` can import it. Only takes ~5 seconds.


In [0]:
%pip install holidays -q
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# 06 — Live API ingestion (AviationStack)

Fetches a live flight, validates against the Silver contract, and writes
`api_bronze_flights` and `api_silver_flights`. Falls back to a committed
fixture when `USE_FIXTURE=True`, so the pipeline stays demoable when the
key is missing or the 100-request free quota is spent.

In [0]:
import sys
sys.path.append("..")

from pyspark.sql.functions import current_timestamp, lit

from src import config
from src.api_pipeline import (
    LookupParams, fetch_flights, load_fixture, project_to_silver, validate_schema,
)

## Runtime switches

In [0]:
dbutils.widgets.dropdown("USE_FIXTURE", "false", ["true", "false"])
dbutils.widgets.text("DEP_IATA", "ATL")
dbutils.widgets.text("ARR_IATA", "LAX")
dbutils.widgets.text("AIRLINE_IATA", "")

USE_FIXTURE = dbutils.widgets.get("USE_FIXTURE").lower() == "true"
params = LookupParams(
    dep_iata=dbutils.widgets.get("DEP_IATA") or None,
    arr_iata=dbutils.widgets.get("ARR_IATA") or None,
    airline_iata=dbutils.widgets.get("AIRLINE_IATA") or None,
)

## Fetch

In [0]:
if USE_FIXTURE:
    payload = load_fixture()
    print(f"Using fixture ({len(payload.get('data', []))} records)")
else:
    access_key = dbutils.secrets.get(
        scope=config.AVIATIONSTACK_SECRET_SCOPE,
        key=config.AVIATIONSTACK_SECRET_KEY,
    )
    payload = fetch_flights(params, access_key)
    print(f"AviationStack returned {len(payload.get('data', []))} records")

Using fixture (2 records)


## Write API Bronze (raw JSON payload as string)

In [0]:
import json as _json

raw_row = [(_json.dumps(payload),)]
bronze_df = (
    spark.createDataFrame(raw_row, ["raw_payload"])
    .withColumn("ingested_at", current_timestamp())
    .withColumn("used_fixture", lit(USE_FIXTURE))
)
(
    bronze_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.API_BRONZE)
)
print(f"Appended to {config.API_BRONZE}")

Appended to workspace.flights.api_bronze_flights


## Project to Silver + DQ check

In [0]:
silver_pdf = project_to_silver(payload)
report = validate_schema(silver_pdf)
print(report)

if not report["passed"]:
    raise ValueError(f"Data-quality gate failed: {report}")

silver_sdf = (
    spark.createDataFrame(silver_pdf)
    .withColumn("ingested_at", current_timestamp())
)
(
    silver_sdf.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.API_SILVER)
)
print(f"Appended {silver_sdf.count()} rows to {config.API_SILVER}")

{'checked_at': '2026-09-01T01:04:55.723485+00:00', 'row_count': 2, 'missing_columns': [], 'null_rate': {'crs_dep_time': 0.0, 'airline_code': 0.0, 'destination_airport_code': 0.0, 'flight_date': 0.0, 'origin_airport_code': 0.0, 'crs_arr_time': 0.0}, 'passed': True}
Appended 2 rows to workspace.flights.api_silver_flights


## Log DQ result to the data-quality table

In [0]:
from pyspark.sql import Row

dq_row = spark.createDataFrame([
    Row(
        checked_at=report["checked_at"],
        source="aviationstack",
        used_fixture=USE_FIXTURE,
        row_count=report["row_count"],
        passed=report["passed"],
        missing_columns=",".join(report["missing_columns"]),
    )
])
(
    dq_row.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(config.DATA_QUALITY_LOG)
)